In [ ]:
import pandas as pd

# =========================
# 1. LOAD DATA RAW
# =========================
df_awal = pd.read_excel(
    '/content/Dataset.xlsx',
    sheet_name='Pangan 16-26',
    header=2
)


df_awal = df_awal[
    (df_awal['Tahun'] <= 2025)
].copy()

df_awal = df_awal[
    (df_awal['Komoditas'] == "PADI")
].copy()

# =========================
# 2. DATA RAW
# =========================
raw_count = len(df_awal)

# =========================
# 3. SELEKSI JENIS OPT
# =========================
opt_list = ['BLAS', 'HDB', 'PBP', 'TIKUS', 'WBC', 'TUNGRO']

df_opt = df_awal[
    df_awal['Jenis OPT'].isin(opt_list)
].copy()

opt_count = len(df_opt)

# =========================
# 4. PILIH KOLOM YANG DIGUNAKAN (Desa dihapus)
# =========================
df_used = df_opt[
    [
        'Tahun',
        'Bulan',
        'Kecamatan',
        'Jenis OPT',
        'ltsP',
        'ltsJ',
        'lpJ',
        "MT",
        "LT",
        "Umur"
    ]
].copy()

# =========================
# 5. KONVERSI NUMERIK
# =========================
df_used['ltsP'] = pd.to_numeric(
    df_used['ltsP'],
    errors='coerce'
).fillna(0)
df_used['LT'] = pd.to_numeric(
    df_used['LT'],
    errors='coerce'
).fillna(0)
df_used['ltsJ'] = pd.to_numeric(
    df_used['ltsJ'],
    errors='coerce'
).fillna(0)

df_used['Umur'] = pd.to_numeric(
    df_used['Umur'],
    errors='coerce'
).fillna(0)
df_used['lpJ'] = pd.to_numeric(
    df_used['lpJ'],
    errors='coerce'
).fillna(0)

# =========================
# 6. HAPUS BARIS DENGAN KOLOM KUNCI KOSONG (Desa dihapus dari subset)
# =========================
df_clean = df_used.dropna(
    subset=[
        'Tahun',
        'Bulan',
        'Kecamatan',
        'Jenis OPT',
    ]
).copy()
clean_count = len(df_clean)

# =========================
# 7. BUAT KOLOM WAKTU
# =========================
df_clean['Waktu'] = pd.to_datetime(
    df_clean['Tahun'].astype(str)
    + '-'
    + df_clean['Bulan'].astype(str)
    + '-01'
)

# =========================
# 8. AGREGASI DATA BULANAN TINGKAT KECAMATAN (Desa dihapus dari groupby)
# =========================
df_group = df_clean.groupby(
    [
        'Tahun',
        'Bulan',
        'Kecamatan',
        'Jenis OPT',
    ],
    as_index=False
).agg({
    'ltsP': 'sum',
    'ltsJ': 'sum',
    # 'lpJ': 'sum',
    'LT': 'sum',   # atau 'max'
    'MT': 'first',
    # 'Umur': 'max'
})

group_count = len(df_group)

# =========================
# 9. DATA FINAL
# =========================
df_final = df_group.copy()
final_count = len(df_final)

# =========================
# 10. TABEL TAHAP PREPROCESSING
# =========================
tahap_preprocessing = pd.DataFrame({
    'Tahap': [
        'Data Raw',
        'Seleksi Jenis OPT',
        'Pembersihan Data (Kecamatan Tidak Kosong)',
        'Agregasi Bulanan (Tingkat Kecamatan)',
        'Data Akhir Digunakan'
    ],
    'Jumlah Record': [
        raw_count,
        opt_count,
        clean_count,
        group_count,
        final_count
    ]
})

print("\n=== TAHAP PREPROCESSING DATA ===\n")
print(tahap_preprocessing.to_string(index=False))

# =========================
# 11. DETAIL DATASET RAW VS FINAL (Jumlah Desa dihapus)
# =========================
detail_dataset = pd.DataFrame({
    'Dataset': [
        'Data Raw',
        'Data Digunakan'
    ],
    'Jumlah Record': [
        raw_count,
        final_count
    ],
    'Jumlah Kecamatan': [
        df_awal['Kecamatan'].nunique(),
        df_final['Kecamatan'].nunique()
    ],
    'Jumlah Jenis OPT': [
        df_awal['Jenis OPT'].dropna().nunique(),
        df_final['Jenis OPT'].nunique()
    ],
    'Jumlah Variabel': [
        len(df_awal.columns),
        len(df_final.columns)
    ]
})

print("\n=== DETAIL DATASET ===\n")
print(detail_dataset.to_string(index=False))

# =========================
# MASTER DATA (Daftar Desa dihapus)
# =========================
daftar_opt = pd.DataFrame({
    'Jenis OPT': sorted(df_final['Jenis OPT'].unique())
})

daftar_kecamatan = pd.DataFrame({
    'Kecamatan': sorted(df_final['Kecamatan'].unique())
})

print(df_awal.info())
# =========================
# 12. SIMPAN KE EXCEL
# =========================
with pd.ExcelWriter('Dataset_Preprocessing_Kecamatan.xlsx') as writer:

    # Ringkasan
    detail_dataset.to_excel(
        writer,
        sheet_name='Detail Dataset',
        index=False
    )

    # Tahap preprocessing
    tahap_preprocessing.to_excel(
        writer,
        sheet_name='Tahap Preprocessing',
        index=False
    )

    # Data final (tingkat kecamatan)
    df_final.to_excel(
        writer,
        sheet_name='Data Final',
        index=False
    )

    # Master data
    daftar_opt.to_excel(
        writer,
        sheet_name='Daftar OPT',
        index=False
    )

    daftar_kecamatan.to_excel(
        writer,
        sheet_name='Daftar Kecamatan',
        index=False
    )

print("File berhasil disimpan: Dataset_Preprocessing_Kecamatan.xlsx")